# DiscoCH in near real time (NRT)
Runs the near-real-time pipeline for one season: transfer → interpolation → model → masking → statistics → maps.

- Each 5-day date *d* is interpolated from the observations up to *d* + `lag_days` and is produced once *d* + `lag_days` ≤ `date_to_run_to`.
- Dates already produced are kept (frozen), so a rerun only adds the dates that have become due. Operationally, set `date_to_run_to = date.today()` and run the whole notebook, e.g. daily.
- The model runs in `model_months`; before the first model month only the interpolation (and the running min/max) advance, and the notebook stops after Masking.

Outputs: `output/DiscoCH/<year>/nrt/`. `DiscoCH_Retro_NRTminmax.ipynb` uses this run's per-date min/max (`model/minmax_by_date/`).

In [ ]:
# User defined parameters
# Paths are machine-specific and come from config.yaml in the project root
# (copy config.example.yaml to config.yaml and set the paths for your machine).
import calendar
from datetime import date
from src.disco_ch_operational.config import load_config

cfg = load_config()

# Run up to this date: date.today() operationally (never a date in the future), or a past date
# to reproduce a run (e.g. date(2026, 10, 5)). Dates are produced up to date_to_run_to - lag_days.
date_to_run_to = date.today()
lag_days = 5                   # days of data after each date it may use (and wait before it is produced)
year = date_to_run_to.year     # the season follows from the date

# Canonical VI name -> token used in the FORCE file names
vi_keys = {
    "NDVI": "NDVI",
    "EVI": "EVI",
    "NDMI": "NDMI",
    "CIRE": "CIre",
    "CCI": "CCI",  # GRVI as listed in the manuscript
}

# Interpolation (FORCE ensemble RBF) and model
int_day = 5                    # one output date every 5 days from 1 January
rbf_sigma = (10, 20, 30)       # kernel widths (days)
rbf_cutoff = 0.95              # share of each kernel's area kept
model_months = (6, 7, 8, 9)    # the model runs June-September (the min/max updates every date)
threshold = 4000               # stored value (0-10000) at/above which a pixel is discolored: 4000 = 40 %

# Earlier season to compare with (Bloom et al. 2026 mosaics)
compare_year = 2018
mosaics_compare = cfg.require("mosaics_2018")
compare_pattern = "Proba_mosaic_*.tif"

max_workers = 10               # tiles processed in parallel

In [ ]:
# Libraries
import os
import glob
import matplotlib.pyplot as plt

from src.disco_ch_operational.config import run_dirs, map_template
from src.disco_ch_operational.transfer import transfer
from src.disco_ch_operational.interpolate import interpolate_tiles, check_data_currency
from src.disco_ch_operational.apply_model import apply_discoch_tiles
from src.disco_ch_operational.mask import mask_and_scale_dir
from src.disco_ch_operational.utils import validate_cog_dir
from src.disco_ch_operational.disco_statistics import summarize_discoloration
from src.disco_ch_operational.compare import plot_year_comparison, BIOGEO_NAMES_EN, BIOGEO_COLORS
from src.disco_ch_operational.clear_sky import (count_clear_sky_tiles, mosaic_clear_sky,
                                                summarize_clear_sky, plot_clear_sky)
from src.disco_ch_operational.hexmap import plot_hex_maps
from src.disco_ch_operational.plot_date_grid import plot_date_grid

# Standard light theme for plots, white figures
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# Machine-specific inputs (config.yaml)
network_directory = cfg["force_network_dir"]  # FORCE VI data on the network
force_directory = cfg["force_dir"]            # local copy of the FORCE VI data
force_tiles_to_process = cfg["tiles_csv"]     # tiles to process (Tile_ID column)
disco_model = cfg["model_path"]               # pre-trained discoloration model
mask_path = cfg["mask_path"]                  # forest mask (EPSG:2056, 1 = keep)
districts = cfg.require("districts")          # political districts (field bzname), also the Swiss outline
biogeo_regions = cfg.require("biogeo_regions")  # biogeographic regions (field DERegionNa)
example_extent = cfg["example_extent"]        # example area for the date grid (Herbetswil)

# Model months as dates: first and last day of the model season
season_start = date(year, model_months[0], 1)
season_end = date(year, model_months[-1], calendar.monthrange(year, model_months[-1])[1])

# Output folders: output/DiscoCH/<year>/nrt/ (interpolated, model, masked, clear_sky, statistics, maps)
dirs = run_dirs(cfg["output_dir"], year, "nrt")
for name, path in dirs.items():
    print(f"{name:13s} {path}")

## Transfer data

In [ ]:
# Copy the season's VI files from the network to the local drive. Only new or changed
# files are copied (same size and time = up to date), so this is quick when nothing is new.
transfer(network_directory, str(year), force_directory, dry_run=True)  # dry_run=False to copy

## Interpolate

In [ ]:
# Is the data on disk current? Warns if tiles have no acquisition after date_to_run_to - lag_days
# (dates produced now would be frozen without data that may still arrive on the share).
newest = check_data_currency(force_directory, year, as_of=date_to_run_to, lag_days=lag_days,
                             vi_token=vi_keys["EVI"], tiles_csv=force_tiles_to_process)

# Every date d with d + lag_days <= date_to_run_to, each from the observations up to d + lag_days.
# Dates already written are kept; only the newly due ones are added.
results = interpolate_tiles(
    tss_dir=force_directory,
    out_dir=dirs["interpolated"],
    vi_keys=vi_keys,
    date_range=(date(year, 1, 1), date(year, 12, 31)),
    as_of=date_to_run_to,
    lag_days=lag_days,
    int_day=int_day,
    rbf_sigma=rbf_sigma, rbf_cutoff=rbf_cutoff,
    despike=False,
    tiles_csv=force_tiles_to_process,  # only these Tile_IDs are processed
    skip_existing=True,                # keep dates already written, only add new ones
    chunk_size=512,
    max_workers=max_workers,
)

for tile_id, by_vi in results.items():
    for vi, (tss_path, out_path, status) in by_vi.items():
        if status.startswith("failed"):
            print(f"{tile_id}/{vi}: {status}")

## Apply Discoloration Model

In [ ]:
# Stepwise normalization of the 5 VIs (running min/max from 1 January up to each date) and the
# pre-trained model, in model_months; then a mosaic per date. Only dates not yet processed are run.
summary = apply_discoch_tiles(
    rbf_dir=dirs["interpolated"],
    out_dir=dirs["model"],           # tiles, mosaic/, minmax/, vi_norm/, minmax_by_date/, logs/
    year=year,
    model_path=disco_model,
    vi_keys=vi_keys,
    tiles_csv=force_tiles_to_process,
    forest_mask_path=None,           # mask raster (1 = keep) to skip non-forest pixels; None = all
    start_date=date(year, 1, 1),     # min/max uses the season from here on
    model_months=model_months,
    export_normalized_vi=True,       # per-date normalized VIs -> vi_norm/
    export_minmax=True,              # per-date min/max snapshots -> minmax_by_date/ (DiscoCH_Retro_NRTminmax)
    export_start_date=season_start,
    export_end_date=season_end,
    mosaic=True,
    plot_mosaic=False,               # plot each newly mosaicked date
    max_workers=max_workers,
)

## Masking

In [ ]:
# Reproject the probability mosaics to the forest mask grid (EPSG:2056), blank outside the
# mask, and write int16 COGs (0-10000, band scale 0.0001; DEFLATE + predictor 2, 512 px tiles,
# overviews to ~1/256). Only new or changed dates are redone.
status = mask_and_scale_dir(input_dir=dirs["mosaic"],
                            mask_path=mask_path,  # None = only reproject to EPSG:2056 and scale
                            output_dir=dirs["masked"],
                            pattern='Disco_Proba_*.tif',
                            name_template=map_template("nrt"),  # -> DiscoCH_NRT_<date>.tif
                            threads=4,
                            )

# Second output: the same scaled COGs without the forest mask (every pixel, same swissEO grid
# and scaling as masked/), to clip as needed later. About 1.3 GB per date.
status_unclipped = mask_and_scale_dir(input_dir=dirs["mosaic"],
                                      mask_path=None,
                                      template_path=mask_path,  # the mask's grid, without masking
                                      output_dir=dirs["unclipped"],
                                      pattern='Disco_Proba_*.tif',
                                      name_template=map_template("nrt"),  # -> DiscoCH_NRT_<date>.tif
                                      threads=4,
                                      )

# Check the COGs: real COG layout (rio-cogeo), 512 x 512 tiles, DEFLATE + predictor 2,
# overviews down to ~1/256, nodata set; prints any file that fails
validate_cog_dir(dirs["masked"], "DiscoCH_NRT_*.tif")
validate_cog_dir(dirs["unclipped"], "DiscoCH_NRT_*.tif")

maps = sorted(glob.glob(os.path.join(dirs["masked"], "DiscoCH_NRT_*.tif")))
print(f"{len(maps)} discoloration map(s); latest: {os.path.basename(maps[-1]) if maps else 'none'}")
if not maps:
    raise RuntimeError(f"No discoloration maps for {year} yet: the model runs in months {model_months}. "
                       "Nothing further to do until then.")

## Spatiotemporal Statistics

In [ ]:
# Per date: discolored share and hectares for Switzerland ("All") and each district
stats = summarize_discoloration(dirs["masked"], threshold, zones=districts, zone_field="bzname",
                                year=year, output_dir=dirs["statistics"], max_workers=8)

In [ ]:
# Compare the season with {compare_year}, aligned by month/day
comparison = plot_year_comparison(
    {
        str(compare_year): dict(mosaic_dir=mosaics_compare, pattern=compare_pattern, year=compare_year),
        str(year): stats,              # already counted above
    },
    threshold=threshold,
    output_dir=dirs["statistics"],
    output_name=f"discoloration_{compare_year}_vs_{year}_nrt",
    max_workers=8,
)

In [ ]:
# Nationally (A) and per biogeographic region (B), in the region colours
comparison_biogeo = plot_year_comparison(
    {
        str(compare_year): dict(mosaic_dir=mosaics_compare, pattern=compare_pattern, year=compare_year),
        str(year): dict(mosaic_dir=dirs["masked"], year=year),   # counted again, by region
    },
    threshold=threshold,
    zones=biogeo_regions, zone_field="DERegionNa",
    region_names=BIOGEO_NAMES_EN, region_colors=BIOGEO_COLORS,
    layout="summary",
    linestyles={str(compare_year): "--", str(year): "-"},
    output_dir=dirs["statistics"],
    output_name=f"discoloration_{compare_year}_vs_{year}_nrt_biogeo",
    max_workers=8,
)

## Clear-Sky Observations

In [ ]:
# Cumulative clear-sky observations per forest pixel, from the raw (uninterpolated) EVI tiles:
# for each 5-day date, the acquisition dates since 1 January up to date + lag_days with a valid EVI value
# (the data the near-real-time interpolation of that date can use). Dates already written are kept.
count_clear_sky_tiles(
    tss_dir=force_directory,
    out_dir=dirs["clear_sky"],
    year=year,
    as_of=date_to_run_to,
    lag_days=lag_days, int_day=int_day,
    export_start_date=season_start,  # rasters for the model season (counts still start 1 January)
    export_end_date=season_end,
    vi_token=vi_keys["EVI"],
    tiles_csv=force_tiles_to_process,
    max_workers=max_workers,
)

# Merge the tiles per date onto the forest mask grid: uint8 COGs in clear_sky/masked (255 = nodata)
mosaic_clear_sky(dirs["clear_sky"], mask_path=mask_path, threads=8)

# Mean and SD across forest pixels per date, national and per biogeographic region
clear_sky = summarize_clear_sky(os.path.join(dirs["clear_sky"], 'masked'),
                                zones=biogeo_regions, zone_field="DERegionNa",
                                year=year, output_dir=dirs["statistics"], max_workers=8)

plot_clear_sky({str(year): clear_sky}, region_names=BIOGEO_NAMES_EN, region_colors=BIOGEO_COLORS,
               lag_days=lag_days, output_dir=dirs["statistics"])

## Hexagon Maps

In [ ]:
# End-of-month hexagon maps, the comparison season vs this one (also saves
# hex_<label>_t<threshold>.gpkg with one proportion field per date)
hexes = plot_hex_maps(
    {
        str(compare_year): dict(mosaic_dir=mosaics_compare, pattern=compare_pattern, year=compare_year),
        str(year): dict(mosaic_dir=dirs["masked"], year=year),
    },
    threshold=threshold,
    hex_area_km2=3,          # hexagon size
    boundary=districts,      # grid built over and clipped to Switzerland
    regions=biogeo_regions,  # biogeographic region boundaries drawn on top
    months=model_months,     # one row per month (its last date)
    output_dir=dirs["maps"],
    max_workers=8,
)

## Example Area: Every Date

In [ ]:
# Every date of the season in the example area, on the grey swisstopo map. The shapefile
# only sets the extent (nothing is clipped); colours dark green (0) to yellow (1).
extent_name = os.path.splitext(os.path.basename(example_extent))[0]
plot_date_grid(
    dataset="nrt",
    mosaic_dir=dirs["masked"],
    extent=example_extent,
    year=year,
    months=model_months,
    alpha=0.6,               # opacity of the probability (0-1)
    min_proba=0,             # lower probabilities not drawn; None = all
    output=os.path.join(dirs["maps"], f"date_grid_{extent_name}_nrt.png"),
)